# Build a Simple Neural Network

In [287]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import tensorflow as tf 

## Define some functions

In [2]:
def relu(x):
    # relu function
    return np.maximum(x, 0)

In [3]:
# test relu
relu(np.array([[-2, 3],
              [5, -3]]))

array([[0, 3],
       [5, 0]])

In [4]:
def cost_J(u, y):
    # MSE cost function: 1 / 2 * ||u-y||^2
    return 0.5*(np.sum((u - y)**2))

In [5]:
# test cost
cost_J(np.array([1,2]), np.array([2,3]))

np.float64(1.0)

## train the weights
The XOR (Exclusive OR) function is a logical operation that returns TRUE if exactly one of the inputs is true, but FALSE if both are true or both are false

In [28]:
# XOR function
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([0, 1, 1, 0]).reshape([-1, 1])
# "-1" above means automatically compute the other dimension
print(X)
print(y)

[[0 0]
 [0 1]
 [1 0]
 [1 1]]
[[0]
 [1]
 [1]
 [0]]


In [57]:
n, p = X.shape

# size (number of nodes) of hidden layer
p1 = 5
lr = 1e-1
max_iter = 30

# initialization
np.random.seed(2)
W1 = np.random.randn(p, p1) # shape is p by p1
W2 = np.random.randn(p1, 1)

for i in range(max_iter):
    # forward propargation (usually should be a for loop, iterate over how many hidden layers there are)
    u1 = X@W1
    h = relu(u1)
    u2 = h@W2

    # compute cost
    J = cost_J(u2, y)
    #print(J)

    # backward propagation for computing gradient (usually a for loop iterate over # of hidden layers)
    du2 = u2 - y
    dW2 = (h.T)@du2
    dh = du2@W2.T
    # du1 = dh*r'(u1): if u1(i,j) > 0, then r'(u1(i,j))=1; if u1(i,j) <= 0, then r'(u1(i,j))=0
    du1 = dh*relu(np.sign(u1))
    dW1 = X.T@du1
    
    # gradient descent
    W1 = W1 - lr * dW1
    W2 = W2 - lr * dW2
print(u2)

[[0.        ]
 [0.96304936]
 [0.93336843]
 [0.15632266]]


## Try train this with linear regression: fail

In [20]:
lm = LinearRegression().fit(X,y)
print(lm.predict(X))
print(y)

[[0.5]
 [0.5]
 [0.5]
 [0.5]]
[[0]
 [1]
 [1]
 [0]]


## write a function to do simple NN

In [272]:
def simple_NN(X, y, seed = 2, lr = 1e-1, hidden_size = 5, max_iter = 30):
    '''
    X, y should be numpy arrays
    '''
    n, p = X.shape
    # reshape y to a column
    y = y.reshape([-1, 1])

    # size (number of nodes) of hidden layer
    p1 = hidden_size
    
    # initialization
    np.random.seed(seed)
    W1 = np.random.randn(p, p1) 
    W2 = np.random.randn(p1, 1)
    
    for i in range(max_iter):
        # forward propargation (usually should be a for loop, iterate over how many hidden layers there are)
        u1 = X@W1
        h = relu(u1)
        u2 = h@W2
    
        # compute cost
        J = cost_J(u2, y)
        #print(J)
    
        # backward propagation for computing gradient (usually a for loop iterate over # of hidden layers)
        du2 = u2 - y
        dW2 = (h.T)@du2
        dh = du2@W2.T
        # du1 = dh*r'(u1): 
        # if u1(i,j) > 0, then r'(u1)(i,j)=1; if u1(i,j) <= 0, then r'(u1)(i,j)=0
        du1 = dh*relu(np.sign(u1)) # point-wise multiplication
        dW1 = X.T@du1
        
        # gradient descent
        W1 = W1 - lr * dW1
        W2 = W2 - lr * dW2
    return W1, W2, J, u2

In [273]:
# works on XOR
simple_NN(X,y)

(array([[-0.41675785, -0.43816504, -2.1361961 ,  1.24168812, -1.79343559],
        [-0.84174737,  0.43270045, -1.24528809, -1.15730864, -0.90900761]]),
 array([[0.55145404],
        [2.31106759],
        [0.04153939],
        [0.77629113],
        [0.53905832]]),
 np.float64(0.003320162025686291),
 array([[0.        ],
        [0.99999995],
        [0.96052743],
        [0.07128983]]))

## Let's try to use our NN to do sale price prediction

In [284]:
# data prep
ames = pd.read_csv("../data/ames.csv")
ames_num = ames.select_dtypes(exclude=['object'])
yh = ames['Sale_Price']
Xh = ames_num.iloc[:,:-3] # selecting all but the last 3 columns
X_train, X_test, y_train, y_test = train_test_split(Xh, yh, test_size = 0.25, random_state = 45)
X_train_n = X_train.to_numpy()
y_train_n = y_train.to_numpy()
X_test_n = X_test.to_numpy()
y_test_n = y_test.to_numpy()

### This is the result of linear regression

In [278]:
lm_h = LinearRegression().fit(X_train,y_train)

In [281]:
# use a data frame for output
results = pd.DataFrame({'training_RMSE':[],
                        'testing_RMSE':[]})
results.loc['linear',:] = [root_mean_squared_error(lm.predict(X_train), y_train.to_numpy()),
                           root_mean_squared_error(lm.predict(X_test), y_test.to_numpy())]
results

,training_RMSE,testing_RMSE
linear,35387.072625,39607.455308


### Use Simple NN

In [282]:
W1, W2, J, u2 = simple_NN(X_train_n, y_train_n, hidden_size = 6, seed = 3, lr = 1e-9, max_iter = 40)